# STGAN su ERA5: il quadro del modello in 5 figure

Run della tesi: **good-sweep-2** (sweep `913ez3rt`, modello dell'ultima epoca, come nel codice degli autori).
Score principale: **G + climatologia** (errore del generatore confrontato con il suo valore tipico per cella, mese e ora).

Tutto ciò che si sceglie (score, scale, climatologia, soglie del 5% e dell'1%, limiti delle classi di fiducia) è fissato
sugli anni di **validazione (2000–2004)**; il **test (2005–2025)** conferma. Ogni figura mostra i due periodi.

Il notebook non ricalcola niente: legge i log dei controlli in `outputs/analisi_good2/` sul server. Sotto ogni figura
è scritto il giudice usato:
- **E-OBS**: stazioni al suolo, indipendenti da ERA5 (giornaliere, solo terraferma europea);
- **CAPE**: variabile ERA5 che nessuno score legge (instabilità convettiva);
- **area di `simple_max`**: dove si trova l'anomalia osservata dei campi ERA5 (stessi dati del modello, non indipendente).

In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path(os.environ['PHYSIQ_ROOT']) if os.environ.get('PHYSIQ_ROOT') else next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'physiq_pv').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Apri il notebook dalla cartella del repository physiq_pv')
sys.path.insert(0, str(ROOT))
from scripts import era5_summary_logs as logs

LOGS = Path(os.environ.get('ANALISI_DIR', ROOT / 'outputs' / 'analisi_good2'))
FIGURES = LOGS / 'figure_relatore'
FIGURES.mkdir(parents=True, exist_ok=True)

def read(name):
    path = LOGS / name
    if not path.exists():
        raise FileNotFoundError(f'Manca il log {path}')
    return path.read_text(encoding='utf-8', errors='replace')

# Colori (palette di riferimento, ordine fisso): modello, score semplice, insieme; grigio per il caso.
MODEL, SIMPLE, BOTH, NEUTRAL = '#2a78d6', '#eb6834', '#1baf7a', '#8a8984'
PERIODS = {'val': 'validazione 2000–2004', 'test': 'test 2005–2025'}
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.25, 'font.size': 10})

def save(fig, name):
    fig.savefig(FIGURES / name, bbox_inches='tight', dpi=200)
    print('salvata', FIGURES / name)

## 1. Perché lo score G + climatologia

Per ogni indicatore delle stazioni: quante volte una giornata segnalata **solo** da quello score è insolita alle stazioni,
rispetto a tutte le giornate (×1 = caso; barra d'errore = intervallo al 95% su anni ricampionati). La scelta è stata
fatta sulla validazione con un criterio fissato prima: indicatore confermato se l'intervallo sta tutto sopra 1.

In [ ]:
INDICATORS = ['rain', 'pressure low', 'pressure drop', 'radiation low', 'heat', 'cold']
LABELS = {'rain': 'pioggia', 'pressure low': 'pressione\nbassa', 'pressure drop': 'calo di\npressione',
          'radiation low': 'radiazione\nbassa', 'heat': 'caldo', 'cold': 'freddo'}

def score_of(name):
    return 'G + climatologia' if 'gen_clim' in name else 'score semplice' if 'simple_max' in name else 'D+G (paper)'

def eobs_table(text):
    table = {}
    for run in logs.eobs(text):
        if 'alone' in run['ratios'] and run['indicator'] in INDICATORS:
            table[(score_of(run['first']), run['indicator'])] = run['ratios']['alone']['either']
    return table

validation = eobs_table(read('eobs_validazione_tre_score.log'))
test = eobs_table(read('calcoli_finali_good2.log'))
scores = [s for s in ('G + climatologia', 'D+G (paper)', 'score semplice') if any(k[0] == s for k in validation)]
colors = {'G + climatologia': MODEL, 'D+G (paper)': NEUTRAL, 'score semplice': SIMPLE}

fig, ax = plt.subplots(figsize=(10, 4.2))
width = 0.8 / len(scores)
x = np.arange(len(INDICATORS))
for k, score in enumerate(scores):
    values = [validation.get((score, i), (np.nan,) * 3) for i in INDICATORS]
    ratio = np.array([v[0] for v in values]); low = np.array([v[1] for v in values]); high = np.array([v[2] for v in values])
    position = x + (k - (len(scores) - 1) / 2) * width
    ax.bar(position, ratio, width * 0.9, color=colors[score], label=f'{score} (validazione)')
    ax.errorbar(position, ratio, yerr=[ratio - low, high - ratio], fmt='none', ecolor='#333', capsize=2, lw=1)
    if score == 'G + climatologia':
        on_test = [test.get((score, i), (np.nan,) * 3)[0] for i in INDICATORS]
        ax.plot(position, on_test, 'D', color='black', ms=5, label='G + climatologia (test)')
ax.axhline(1, color='black', lw=1)
ax.set_xticks(x, [LABELS[i] for i in INDICATORS])
ax.set_ylabel('giornate insolite: solo questo score / tutte')
confirmed = lambda table, score: sum(table.get((score, i), (np.nan,) * 3)[1] > 1 for i in INDICATORS)
ax.set_title(f'Stazioni E-OBS: G + climatologia confermato su {confirmed(validation, "G + climatologia")} indicatori '
             f'su 6 (validazione), {confirmed(test, "G + climatologia")} su 6 (test)')
ax.legend(frameon=False, fontsize=8, ncol=2)
save(fig, '1_scelta_score_eobs.png')
plt.show()
for (score, indicator), (ratio, low, high) in sorted(validation.items()):
    print(f'{score:18s} {LABELS[indicator].replace(chr(10), " "):18s} val {ratio:.2f} ({low:.2f}-{high:.2f})', 'confermato' if low > 1 else '')

*Giudice: stazioni E-OBS (indipendenti). Segnalato al 5%; "solo questo score" = nessuno degli altri due lo segnala
quel giorno. D+G è lo score combinato del paper STGAN: le sue giornate esclusive sono più calme del normale.*

## 2. Cosa vede in più il modello

Quasi tutti i segnali che dà **solo il modello** (27% dei suoi segnali) sono a 1 cella da un evento visto dagli score
semplici: il **bordo** degli eventi. Confronto con il bordo che nessuno segnala, alla stessa distanza dagli stessi eventi:
che tipo di evento è?

In [ ]:
TYPES = ['mixed', 'heat', 'cold', 'wind', 'rain', 'convective', 'other']
TYPE_LABELS = {'mixed': 'composto\n(più segnali)', 'heat': 'caldo', 'cold': 'freddo', 'wind': 'vento', 'rain': 'pioggia',
               'convective': 'convet-\ntivo', 'other': 'nessun\nsegnale\nforte'}

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, period in zip(axes, ('val', 'test')):
    rows = logs.event_types(read(f'tipi_eventi_{period}_v2.log'))
    alone = next(name for name in rows if name.startswith('edge,') and name.endswith('alone'))
    x = np.arange(len(TYPES))
    for k, (name, label, color) in enumerate(((alone, 'bordo visto solo dal modello', MODEL),
                                             ('edge, no score', 'bordo che nessuno segnala', NEUTRAL))):
        ax.bar(x + (k - 0.5) * 0.4, [rows[name][t] for t in TYPES], 0.36, color=color, label=label)
    ax.set_xticks(x, [TYPE_LABELS[t] for t in TYPES], fontsize=8)
    ax.set_title(PERIODS[period])
axes[0].set_ylabel('% delle celle')
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle('Il modello estende gli eventi composti (vento, pioggia, convezione e caldo insieme)')
save(fig, '2_tipi_evento_bordo.png')
plt.show()

*Tipi definiti su variabili ERA5 (raffica > 98° percentile, temperatura > 90° / < 10°, pioggia > 95° delle ore
piovose, CAPE > 95° in un'ora piovosa; percentili per cella, mese e ora). Descrive il tipo di tempo, non conferma
che sia vero: la conferma è nella figura 1.*

## 3. Il modello come allarme

Celle dove la CAPE **non** è insolita adesso: quante volte, nelle 3 ore dopo, diventa insolita (5% più alto) nella stessa
cella? Per ogni score semplice, i quattro casi: segnalano **tutti e due**, **solo il modello**, **solo lo score semplice**,
**nessuno**. Vento e moto passato come allarme = celle dove la zona anomala di `simple_max` arriva fra 3 ore.

In [ ]:
METHODS = {'stagionalita': 'stagionalità', 'variazione3h': 'variazione\n3 h', 'vento': 'vento\n850 hPa',
           'moto': 'moto\npassato'}
CASES = (('both', 'tutti e due', BOTH), ('first only', 'solo modello', MODEL),
         ('second early only', 'solo score semplice', SIMPLE))
parts = logs.blocks(read('confronto_semplici_soglie.log'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, level in zip(axes, (5, 1)):
    x = np.arange(len(METHODS))
    for k, (case, label, color) in enumerate(CASES):
        for period, style in (('test', None), ('val', 'o')):
            values = []
            for method in METHODS:
                rows = logs.precursor(parts[f'{period}, soglia {level}%, {method}, giudice cape 5%'])
                values.append(rows[(0, 3, case)][2])
            position = x + (k - 1) * 0.26
            if style is None:
                ax.bar(position, values, 0.24, color=color, label=label)
            else:
                ax.plot(position, values, style, color='black', ms=3.5, label='validazione' if k == 0 else None)
    neither = [logs.precursor(parts[f'test, soglia {level}%, {m}, giudice cape 5%'])[(0, 3, 'neither')][2] for m in METHODS]
    ax.hlines(neither, x - 0.4, x + 0.4, color=NEUTRAL, lw=2, label='nessuno (caso)')
    ax.set_xticks(x, list(METHODS.values()))
    ax.set_title(f'soglia {level}% (barre: test; punti: validazione)')
axes[0].set_ylabel('% seguiti da CAPE insolita entro 3 h')
handles, names = axes[0].get_legend_handles_labels()
fig.legend(handles, names, loc='lower center', ncol=5, frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.06))
fig.suptitle('Solo il modello è più preciso di ogni score semplice; d\'accordo con lui è il caso migliore')
save(fig, '3_allarmi_cape.png')
plt.show()

*Giudice: CAPE nel suo 5% più alto per cella, mese e ora (variabile ERA5 che nessuno score legge; favorisce il lato
convettivo). Le celle di uno stesso evento non sono indipendenti: le quote sono di celle per passo di 3 ore. Con la
raffica e con "CAPE o raffica" come giudice l'ordine resta lo stesso.*

## 4. La direzione degli eventi

Per ogni evento di `simple_max` (almeno 4 celle): il **bordo che segnala solo il modello** punta dove l'evento va nelle
3 ore dopo? Confronto con il vento a 850 hPa, il moto passato e un anello qualsiasi (solo la forma dell'area), e con i
casi in cui bordo e vento (o bordo e moto) **puntano dalla stessa parte**.

In [ ]:
BARS = (('ring', 'anello\nqualsiasi', NEUTRAL, 'same events'), ('gen_clim only', 'bordo del\nmodello', MODEL, 'same events'),
        ('past motion', 'moto\npassato', SIMPLE, 'same events'), ('wind at 850 hPa', 'vento\n850 hPa', SIMPLE, 'same events'),
        ('past motion', 'bordo +\nmoto\nconcordi', BOTH, 'agreement'),
        ('wind at 850 hPa', 'bordo +\nvento\nconcordi', BOTH, 'agreement'))
parts = logs.blocks(read('dropout_parita_errore.log'))

def right_side(tables, name, table):
    rows = tables[table]
    if table == 'agreement':
        return rows[(name, 'shift', 3)][1]
    key = next(k for k in rows if k[0] == name and k[1:] == ('shift', 3))
    return rows[key][1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), sharey=True)
for ax, level in zip(axes, (5, 1)):
    x = np.arange(len(BARS))
    for period, style in (('test', None), ('val', 'o')):
        tables = logs.direction(parts[f'direzione {period}, soglia {level}%, giudice area simple_max'])
        values = [right_side(tables, name, table) for name, _, _, table in BARS]
        if style is None:
            ax.bar(x, values, 0.6, color=[c for _, _, c, _ in BARS])
            for xi, v in zip(x, values):
                ax.text(xi, v + 0.6, f'{v:.0f}%', ha='center', fontsize=8)
        else:
            ax.plot(x, values, 'o', color='black', ms=3.5, label='validazione')
    ax.axhline(50, color='black', lw=1, ls=':')
    ax.set_xticks(x, [label for _, label, _, _ in BARS], fontsize=8)
    ax.set_ylim(40, 85)
    ax.set_title(f'soglia {level}% (barre: test; punti: validazione)')
axes[0].set_ylabel('% di eventi col lato giusto (3 h)')
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle('Il bordo da solo indica poco; d\'accordo con vento o moto passato si arriva al 72–78%')
save(fig, '4_direzione.png')
plt.show()

*Giudice: spostamento dell'area segnalata da `simple_max` (dove sta l'anomalia osservata dei campi ERA5 tre ore
dopo). Usa gli stessi dati del modello e coincide con l'area da cui si calcola il moto passato: non esiste un giudice
indipendente della direzione (CAPE e raffica non seguono lo spostamento; E-OBS è giornaliero). Linea punteggiata: caso
(50%). "Concordi" = percentuale sugli eventi in cui le due direzioni stanno dalla stessa parte (circa metà).*

## 5. A cosa serve il MC dropout di G: una fiducia per ogni segnale

Le 20 estrazioni del dropout di G danno 20 errori: quanto sono in disaccordo (lo **spread**), a parità di errore, divide
i segnali in 5 classi (limiti fissati sulla validazione, applicati al test). Per ogni classe: quanti segnali sono ancora
un **evento in corso** e quanti un'anomalia **solo nella previsione** di G (tempo già tornato normale).

In [ ]:
GROUPS = (('inside (model and other)', 'modello + score semplici', BOTH),
          ('all flags of the model', 'tutti i segnali del modello', NEUTRAL),
          ('model only', 'solo modello', MODEL))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for period, style in (('test', '-'), ('val', '--')):
    rows = logs.confidence(read(f'fiducia_{period}.log'))
    for label, name, color in GROUPS:
        classes = range(1, 6)
        axes[0].plot(classes, [rows[(label, c)][1] for c in classes], style, marker='o', ms=4, color=color,
                     label=f'{name}' if period == 'test' else None)
        axes[1].plot(classes, [rows[(label, c)][2] for c in classes], style, marker='o', ms=4, color=color)
axes[0].set_title('evento ancora in corso')
axes[1].set_title('anomalia solo nella previsione')
for ax in axes:
    ax.set_xticks(range(1, 6), ['1\nG sicuro', '2', '3', '4', '5\nG in dubbio'])
    ax.set_ylabel('% dei segnali')
axes[0].legend(frameon=False, fontsize=8, title='linea piena: test; tratteggiata: validazione', title_fontsize=8)
fig.suptitle('Più le estrazioni di G sono in disaccordo, meno il segnale è un evento in corso')
save(fig, '5_fiducia_dropout.png')
plt.show()

*Giudice: campi ERA5 osservati ancora oltre 1 deviazione standard dalla norma stagionale ("evento in corso"), oppure
tornati entro 1 mentre la previsione di G resta oltre ("solo previsione"); segnali vicini o dentro gli eventi, 400 istanti.
A parità di errore lo spread di G distingue i due casi con AUC 0.63 (validazione e test); lo spread di D no (circa 0.5),
e per gli allarmi di formazione e la direzione lo spread non aggiunge niente oltre all'errore.*